# Combine Two Microarray Parquet Files

This notebook loads two standardized microarray parquet files, unions their SNP/variant sets by `# rsid`, and reports genotype mismatches on intersecting variants.

It assumes the parquet files contain at least:
- `# rsid` or a similar rsid column
- a genotype column such as `genotype`, `GT`, or `call`
- optional metadata columns like `chromosome` and `position`

In [2]:
import pandas as pd
from pathlib import Path


In [ ]:
file1_path = Path("/home/frederik/github_projects/SNPster/data pipeline/file_combiner_module/test_file/1.chrALL.standardizedMicroarray.parquet")
file2_path = Path("/home/frederik/github_projects/SNPster/data pipeline/file_combiner_module/test_file/2.chrALL.standardizedMicroarray.parquet")
#file2_path = Path("/srv/raw/genome_Craig_Falls_Full_20140720192139.zip")


def load_microarray_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet file: {path}")
    
    data = pd.read_csv(path, sep="\t", low_memory=False, compression='zip' if path.suffix == '.zip' else None, comment='#')
    column_names = ["RSID","CHROMOSOME","POSITION","RESULT"]
    #rename columns to the columns_names list in that order
    data = data.rename(columns={data.columns[0]: column_names[0], data.columns[1]: column_names[1], data.columns[2]: column_names[2], data.columns[3]: column_names[3]})
    return data
    
    
    
columns_names = ["RSID","CHROMOSOME","POSITION","RESULT"]
    
print("Loading microarray data from file1...")
microarray_data1 = pd.read_parquet(file1_path, engine = 'pyarrow')
microarray_data2 = pd.read_parquet(file2_path, engine = 'pyarrow')



microarray_data1.columns = columns_names
microarray_data2.columns = columns_names

microarray_data1['vendor'] = 'MyHeritage'
microarray_data2['vendor'] = '23AndMe'

print(microarray_data1.head())
print(microarray_data2.head())



Loading microarray data from file1...
          RSID CHROMOSOME  POSITION RESULT      vendor
1  rs116587930          1    792461     AG  MyHeritage
2    rs3131972          1    817341     AG  MyHeritage
3   rs12184325          1    818725     CC  MyHeritage
5  rs114525117          1    823656     GG  MyHeritage
6   rs12124819          1    841166     AA  MyHeritage
          RSID CHROMOSOME  POSITION RESULT   vendor
1  rs116587930          1    792461     GG  23AndMe
2    rs3131972          1    817341     GG  23AndMe
3   rs12184325          1    818725     CC  23AndMe
5  rs114525117          1    823656     GG  23AndMe
6   rs12127425          1    858952     GG  23AndMe


In [ ]:
# Filter both datasets identically
data1 = microarray_data1.copy()
data2 = microarray_data2.copy()



# Compare only RSIDs present in BOTH files
comparison = data1[["RSID", "RESULT"]].merge(
    data2[["RSID", "RESULT"]],
    on="RSID",
    how="inner",
    suffixes=("_file1", "_file2")
)

n_intersecting_rsids = len(comparison)
total_unique_rsids_full = len(pd.concat([data1["RSID"], data2["RSID"]]).unique())
total_unique_rsids_file1 = len(data1["RSID"].unique())
total_unique_rsids_file2 = len(data2["RSID"].unique())

conflicts = comparison[
    comparison["RESULT_file1"] != comparison["RESULT_file2"]
]

n_conflicting_rsids = len(conflicts)

percent_disagreeing = (
    n_conflicting_rsids / n_intersecting_rsids * 100
)

print(f"Intersecting RSIDs: {n_intersecting_rsids}")
print(f"Conflicting RSIDs: {n_conflicting_rsids}")
print(f"Disagreement: {percent_disagreeing:.2f}%")

#print disagreement df

conflicts = comparison[
    comparison["RESULT_file1"] != comparison["RESULT_file2"]
].sort_values("RSID")

n_conflicting_rsids = len(conflicts)

percent_disagreeing = (
    n_conflicting_rsids / n_intersecting_rsids * 100
)

print(f"Intersecting RSIDs: {n_intersecting_rsids}")
print(f"Conflicting RSIDs: {n_conflicting_rsids}")
print(f"Disagreement: {percent_disagreeing:.2f}%")

print(conflicts.to_string(index=False))

print(f"Total unique RSIDs in file1: {total_unique_rsids_file1}")
print(f"Total unique RSIDs in file2: {total_unique_rsids_file2}")
print(f"Total unique RSIDs across both files: {total_unique_rsids_full}")

Intersecting RSIDs: 560719
Conflicting RSIDs: 146487
Disagreement: 26.12%
Intersecting RSIDs: 560719
Conflicting RSIDs: 146487
Disagreement: 26.12%
       RSID RESULT_file1 RESULT_file2
  rs1000002           TT           CT
  rs1000005           CG           CC
  rs1000007           CT           TT
 rs10000092           TT           CT
 rs10000209           CC           CT
  rs1000026           CT           CC
 rs10000265           GG           AG
 rs10000438           CT           TT
 rs10000451           CC           CT
 rs10000500           TT           GT
  rs1000056           CC           CT
 rs10000663           CT           TT
  rs1000073           AG           GG
 rs10000976           CC           CT
 rs10000991           CC           CT
  rs1000110           CT           CC
 rs10001131           CC           CT
 rs10001148           AG           GG
 rs10001154           CC           CT
  rs1000119           TT           CT
  rs1000121           TT           CC
 rs10001214     

In [ ]:
import pandas as pd
from pathlib import Path

# Update these paths before running.
file1_path = Path("/home/frederik/github_projects/SNPster/data pipeline/harmonizing_module/test_data/IMPID29.chr22.standardizedMicroarray.parquet")
file2_path = Path("/path/to/second_microarray.parquet")
output_path = Path("/home/frederik/github_projects/SNPster/data pipeline/util_dev/ultimate_microarray.parquet")


def load_microarray_parquet(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet file: {path}")
    return pd.read_parquet(path)


def find_identifier_column(df: pd.DataFrame) -> str:
    candidates = ["# rsid", "rsid", "SNP", "snp", "ID", "id"]
    for column in candidates:
        if column in df.columns:
            return column
    raise ValueError(f"Could not find an rsid/SNP identifier column. Available columns: {list(df.columns)}")


def find_genotype_column(df: pd.DataFrame) -> str:
    candidates = ["genotype", "GT", "call", "genotype_call", "allele", "genotype_value"]
    for column in candidates:
        if column in df.columns:
            return column
    raise ValueError(f"Could not find a genotype column. Available columns: {list(df.columns)}")


def normalize_microarray(df: pd.DataFrame, source_label: str) -> pd.DataFrame:
    id_column = find_identifier_column(df)
    genotype_column = find_genotype_column(df)

    normalized = df.copy()
    normalized = normalized.rename(columns={id_column: "variant_id", genotype_column: f"genotype_{source_label}"})
    normalized = normalized.drop_duplicates(subset=["variant_id"], keep="first")
    return normalized


In [ ]:
# Uncomment if your notebook kernel does not have parquet support yet:
# %pip install pyarrow

left = normalize_microarray(load_microarray_parquet(file1_path), "file1")
right = normalize_microarray(load_microarray_parquet(file2_path), "file2")

combined = left.merge(
    right,
    on="variant_id",
    how="outer",
    suffixes=("_file1", "_file2"),
    indicator=True,
)

left_genotype_col = "genotype_file1"
right_genotype_col = "genotype_file2"

combined["genotype_mismatch"] = (
    combined[left_genotype_col].notna()
    & combined[right_genotype_col].notna()
    & (combined[left_genotype_col] != combined[right_genotype_col])
)

discrepancies = combined.loc[
    combined["genotype_mismatch"],
    ["variant_id", left_genotype_col, right_genotype_col, "_merge"]
].copy()

ultimate_microarray = pd.DataFrame({"variant_id": combined["variant_id"]})

all_base_columns = set()
all_base_columns.update(col[:-6] for col in combined.columns if col.endswith("_file1"))
all_base_columns.update(col[:-6] for col in combined.columns if col.endswith("_file2"))
all_base_columns.discard("variant_id")

for column in sorted(all_base_columns):
    file1_column = f"{column}_file1"
    file2_column = f"{column}_file2"

    if file1_column in combined.columns and file2_column in combined.columns:
        ultimate_microarray[column] = combined[file1_column].combine_first(combined[file2_column])
    elif file1_column in combined.columns:
        ultimate_microarray[column] = combined[file1_column]
    elif file2_column in combined.columns:
        ultimate_microarray[column] = combined[file2_column]

ultimate_microarray["genotype"] = combined[left_genotype_col].combine_first(combined[right_genotype_col])
ultimate_microarray["genotype_mismatch"] = combined["genotype_mismatch"]
ultimate_microarray["source"] = combined["_merge"]

sort_columns = [column for column in ["chromosome", "position", "variant_id"] if column in ultimate_microarray.columns]
if sort_columns:
    ultimate_microarray = ultimate_microarray.sort_values(sort_columns, kind="mergesort").reset_index(drop=True)
else:
    ultimate_microarray = ultimate_microarray.sort_values("variant_id", kind="mergesort").reset_index(drop=True)

print(f"File 1 variants: {len(left)}")
print(f"File 2 variants: {len(right)}")
print(f"Union variants: {len(ultimate_microarray)}")
print(f"Intersecting variants: {(combined['_merge'] == 'both').sum()}")
print(f"Genotype mismatches: {len(discrepancies)}")

ultimate_microarray.to_parquet(output_path, index=False)
print(f"Wrote combined parquet to: {output_path}")

discrepancies